## Setup

In [3]:
import sys
sys.path.append("..")

from src.dataset import XRayDataset
from src.model import get_model
from src.train import train_model


%load_ext autoreload
%autoreload 2

## Dataset Check

In [2]:
dataset = XRayDataset(r"..\train")

print(len(dataset))
img, label = dataset[0]
print("label:", label)
img.shape

5216
label: 0


torch.Size([3, 224, 224])

In [3]:
print(dataset[8])

(tensor([[[-1.8610, -2.0494, -2.1179,  ..., -0.6794, -0.4054,  0.1254],
         [-1.7412, -1.9295, -2.0837,  ..., -0.7308, -0.4226,  0.1083],
         [-1.6555, -1.7754, -1.9980,  ..., -0.7822, -0.4739,  0.0912],
         ...,
         [-2.1179, -2.1179, -2.1179,  ..., -2.1179, -2.1179, -2.1179],
         [-2.1179, -2.1179, -2.1179,  ..., -2.1179, -2.1179, -2.1179],
         [-2.1179, -2.1179, -2.1179,  ..., -2.1179, -2.1179, -2.1179]],

        [[-1.7731, -1.9657, -2.0357,  ..., -0.5651, -0.2850,  0.2577],
         [-1.6506, -1.8431, -2.0007,  ..., -0.6176, -0.3025,  0.2402],
         [-1.5630, -1.6856, -1.9132,  ..., -0.6702, -0.3550,  0.2227],
         ...,
         [-2.0357, -2.0357, -2.0357,  ..., -2.0357, -2.0357, -2.0357],
         [-2.0357, -2.0357, -2.0357,  ..., -2.0357, -2.0357, -2.0357],
         [-2.0357, -2.0357, -2.0357,  ..., -2.0357, -2.0357, -2.0357]],

        [[-1.5430, -1.7347, -1.8044,  ..., -0.3404, -0.0615,  0.4788],
         [-1.4210, -1.6127, -1.7696,  ..., -

## Restnet18 structure

In [4]:
import torchvision.models as models
from torchvision.models import ResNet18_Weights

model = models.resnet18(weights=ResNet18_Weights.DEFAULT)
print(model)


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [5]:
print(ResNet18_Weights.DEFAULT.transforms())

ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)


## Dataloader check

In [6]:
from torch.utils.data import DataLoader

dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

for img, labels in dataloader:
    print(img.shape)
    print(labels.shape)
    break

torch.Size([32, 3, 224, 224])
torch.Size([32])


## Forward Pass(Original Model)

In [7]:
from torch.utils.data import DataLoader

dataloader = DataLoader(dataset, batch_size=32, shuffle=True)
for images, labels in dataloader:
    outputs = model(images)
    print(outputs.shape)
    break

torch.Size([32, 1000])


## Modified Model

In [8]:
from src.model import get_model

model = get_model()
print(model.fc)


Linear(in_features=512, out_features=2, bias=True)


## Forward Pass(Modified)

In [9]:
outputs = model(images)
print(outputs.shape)

torch.Size([32, 2])


### Freeze check

In [10]:
from src.model import get_model
model = get_model()

for name, param in model.named_parameters():
    if param.requires_grad:
        print("Trainable:", name)

Trainable: fc.weight
Trainable: fc.bias


## Loss check

In [11]:
from torch.utils.data import DataLoader
import torch.nn as nn

criterion = nn.CrossEntropyLoss()
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

for images, labels in dataloader:
    outputs = model(images)
    loss = criterion(outputs, labels)
    
    print("Loss:", loss.item())
    break

Loss: 1.8282153606414795


## Training pipeline

In [4]:
from main import entry_point
entry_point()


Starting training...
Epoch: 0, Loss: 0.26362252852485224
Epoch: 1, Loss: 0.1552694334985654
Epoch: 2, Loss: 0.14510077835729152
Epoch: 3, Loss: 0.12960116861193824
Epoch: 4, Loss: 0.11976585101816187
Epoch: 5, Loss: 0.12010317278648812
Epoch: 6, Loss: 0.1112169191652646
Epoch: 7, Loss: 0.11085711296167841
Epoch: 8, Loss: 0.1158718783179858
Epoch: 9, Loss: 0.1013710722098687
